# Single Image Processing from a LIF Container

This notebook processes 3D co-culture images from LIF containers with the following steps:

1. Predict 3D cell labels with CellposeSAM (default) for each configured cell channel, including anisotropy correction and optional volume filtering.
2. Clean HPFa_red cell labels by masking out voxels that colocalize with A549_green predictions (label IDs preserved).
3. Compute `%_area_cleaned` per HPFa_red object and remove objects at or above a configurable threshold.

All processing steps are visualized in Napari. Original Cellpose predictions are saved to disk; masking and object filtering are applied in memory afterwards.

Image outputs are written to `RAW_DATA_DIRECTORY/cell_labels/<lif_container_id>/`.

### Environment setup
The next cell sets up imports and makes sure the `src` folder is available on the Python path.

Input needed from researcher: run this as-is unless your project folder layout is different.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_output_dir,
    load_precomputed_results_if_available,
)
from utils.segmentation import (
    predict_nuclei_labels_cp as predict_cell_labels_cp,
    predict_nuclei_labels_vo as predict_cell_labels_vo,
    clean_colocalized_labels,
)

from utils.feature_extraction import extract_nuclei_features_per_marker as extract_cell_features_per_marker

from utils.data_viz import (
    map_df_column_to_labels,
    plot_prop_to_3d_centroids
)

import tifffile
import napari
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

### Configure experiment inputs
Set core paths and model choices used in the workflow.

Input needed from researcher: update `RAW_DATA_DIRECTORY`, `MIN_MAX_CELL_VOLUME`, `CELL_CHANNELS`, `PCT_AREA_CLEANED_THRESHOLD`, marker definitions, and image/container indices for your experiment.

In [ ]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"Z:\margrethe.c.stahl\Co-culture"

# Channel indices used for CellposeSAM-based 3D cell segmentation (one prediction per channel)
CELL_CHANNELS = (0, 1)

# Cell segmentation method ("cellpose", "voronoi-otsu")
CELL_LABELING = "cellpose"

# Tunable Voronoi-Otsu parameters, applies if choosing "voronoi-otsu" above
SPOT_SIGMA = 2      # larger -> fewer separated detected spots
OUTLINE_SIGMA = 2   # larger -> smoother object outlines

# Minimum and maximum cell label volume to use for filtering predicted cell labels.
# Use a (min, max) voxel-count tuple, or None to skip volume filtering.
MIN_MAX_CELL_VOLUME = None #(500, 6500) seems safe after exploring the volume distribution

# Remove HPFa_red objects whose masked area loss is at or above this percent.
PCT_AREA_CLEANED_THRESHOLD = 20

# Marker information
MARKERS = (("A549_green", 0), ("HPFa_red", 1), ("brightfield", 2))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 4

### Discover LIF containers
This cell scans the raw data directory and lists available .lif containers.

Input needed from researcher: confirm the directory contains the expected files.

In [ ]:
# Iterate through the .lif container files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

if not lif_containers:
    raise FileNotFoundError(
        f"No .lif containers found in '{RAW_DATA_DIRECTORY}'. "
        "Check RAW_DATA_DIRECTORY and file extension."
    )

if not 0 <= LIF_CONTAINER_INDEX < len(lif_containers):
    raise IndexError(
        f"LIF_CONTAINER_INDEX={LIF_CONTAINER_INDEX} is out of range. "
        f"Valid range is 0..{len(lif_containers)-1}."
    )

lif_containers

### Select one container and image
Choose the container and image index to analyze, then load metadata and image data.

Input needed from researcher: set LIF_CONTAINER_INDEX and LIF_IMAGE_INDEX to the sample you want to inspect in the cells above.

In [ ]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)


### Display input channels in Napari
This opens a 3D Napari view of all channels to verify image quality and channel order.

Input needed from researcher: visually confirm that channels and marker names match expectations.

In [ ]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['green', 'red', 'grey'],
                name=[marker[0] for marker in MARKERS]
                )

### Predict or load cell labels (CellposeSAM default)
For each channel in `CELL_CHANNELS`, this step checks whether cell label masks already exist. If so, they are loaded and displayed in Napari. Otherwise, new cell labels are predicted—CellposeSAM by default, or Voronoi-Otsu—using anisotropy correction.

- One labels file is written per channel as `{lif_image_name}_{marker_name}_cell_labels.tif`.
- If CellposeSAM is selected, set and verify Z anisotropy and `CELL_CHANNELS`.
- If Voronoi-Otsu is used, also adjust `SPOT_SIGMA` (larger = merges nearby cells) and `OUTLINE_SIGMA` (controls boundary smoothness).

Original Cellpose predictions are saved to disk for reuse; cleaning runs afterwards on the loaded/predicted arrays.

In [ ]:
# Ensure output directory for this container's cell labels
cell_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="cell_labels")
print(f"Cell labels directory: {cell_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Map channel index -> marker name for filename suffixes
channel_to_marker = {idx: name for name, idx in MARKERS}
cell_labels = {}

for cell_channel in CELL_CHANNELS:
    marker_name = channel_to_marker[cell_channel]
    image_id = f"{lif_image_name}_{marker_name}"

    # Load precomputed labels when available; otherwise predict and store them
    labels = load_precomputed_results_if_available(
        cell_labels_dir, image_id, results_type="cell_labels"
    )

    if labels is not None:
        print(f"Predictions already calculated for: {image_id} ...loading")
    else:
        if CELL_LABELING == "voronoi-otsu":
            labels = predict_cell_labels_vo(
                lif_image,
                rescale_factor,
                cell_channel,
                MIN_MAX_CELL_VOLUME,
                SPOT_SIGMA,
                OUTLINE_SIGMA,
                visualize=False,
                viewer=viewer,
            )
        else:
            print(f"Predicting cell labels with Cellpose for {marker_name} (channel {cell_channel})")
            labels = predict_cell_labels_cp(
                lif_image,
                rescale_factor,
                cell_channel,
                MIN_MAX_CELL_VOLUME,
                visualize=False,
                viewer=viewer,
            )

        # Save the original prediction (one file per channel/marker)
        cell_labels_path = cell_labels_dir / f"{image_id}_cell_labels.tif"
        tifffile.imwrite(cell_labels_path, labels)

    cell_labels[marker_name] = labels
    viewer.add_labels(labels, name=f"{marker_name}_cell_labels")



### Clean HPFa_red labels by A549_green colocalization
Mask out HPFa_red voxels that overlap with A549_green cell labels. Label IDs are preserved so `%_area_cleaned` can be computed later by matching objects. Original predictions on disk are left unchanged; only an in-memory cleaned labels image is produced.

In [ ]:
# Clean HPFa_red by zeroing voxels that colocalize with A549_green (IDs preserved)
hpfa_red_cleaned = clean_colocalized_labels(
    labels_to_clean=cell_labels["HPFa_red"],
    colocalizing_labels=cell_labels["A549_green"],
)

viewer.add_labels(hpfa_red_cleaned, name="HPFa_red_cleaned")
n_hpfa = int(np.unique(cell_labels["HPFa_red"]).size - 1)  # exclude background 0
n_hpfa_cleaned = int(np.unique(hpfa_red_cleaned).size - 1)
print(f"HPFa_red objects: {n_hpfa} -> after masking: {n_hpfa_cleaned}")

### Extract cell features
Build image descriptors and compute per-cell morphological and intensity features for A549_green, original HPFa_red, and masked HPFa_red labels.

Input needed from researcher: confirm marker/channel mapping reflects your biological labels.

In [ ]:
# Create a dictionary containing all image descriptors
descriptor_dict = {
            "lif_container_id": lif_container_id,
            "lif_image_name": lif_image_name,
            }

# Extract morphological and intensity features per population
props_a549 = extract_cell_features_per_marker(
    cell_labels["A549_green"], lif_image, MARKERS, {**descriptor_dict, "population": "A549_green"}
)
props_hpfa = extract_cell_features_per_marker(
    cell_labels["HPFa_red"], lif_image, MARKERS, {**descriptor_dict, "population": "HPFa_red"}
)
props_hpfa_cleaned = extract_cell_features_per_marker(
    hpfa_red_cleaned, lif_image, MARKERS, {**descriptor_dict, "population": "HPFa_red_cleaned"}
)

### Compute `%_area_cleaned`
Compare each original HPFa_red object's area to its remaining area after A549 masking. Objects fully removed by masking get `%_area_cleaned = 100`. Label IDs must match between original and masked labels (masking does not relabel).

In [ ]:
# Fraction of each HPFa_red object's area removed by A549 masking.
# Labels absent from props_hpfa_cleaned are treated as fully cleaned (100%).
cleaned_area_by_label = props_hpfa_cleaned.set_index("label")["area"]
area_after_cleaning = props_hpfa["label"].map(cleaned_area_by_label).fillna(0)
pct_area_cleaned = (1.0 - area_after_cleaning / props_hpfa["area"]) * 100.0

if "%_area_cleaned" in props_hpfa.columns:
    props_hpfa.drop(columns=["%_area_cleaned"], inplace=True)

area_col_idx = props_hpfa.columns.get_loc("area")
props_hpfa.insert(area_col_idx + 1, "%_area_cleaned", pct_area_cleaned)

props_hpfa[["label", "area", "%_area_cleaned"]].head()

### Explore `%_area_cleaned` vs A549_green intensity
Joint plot of percent area removed by masking versus mean A549_green intensity in original HPFa_red objects. Use this to choose `PCT_AREA_CLEANED_THRESHOLD`.

In [ ]:
sns.set_style("ticks")

plot_df = props_hpfa[["%_area_cleaned", "A549_green_mean_int"]].dropna()

g = sns.JointGrid(
    data=plot_df,
    x="%_area_cleaned",
    y="A549_green_mean_int",
)
g.plot_joint(sns.scatterplot, alpha=0.5, s=20, edgecolor=None)
g.plot_marginals(sns.kdeplot, fill=True)
g.set_axis_labels("% Area Cleaned", "A549_green mean intensity")
g.figure.suptitle("% Area Cleaned vs A549_green mean intensity", y=1.02)
sns.despine()
plt.show()

### Filter HPFa_red objects by `%_area_cleaned`
Remove original HPFa_red objects whose masked area loss is at or above `PCT_AREA_CLEANED_THRESHOLD`. Disk predictions are unchanged; the filtered labels image is kept in memory for downstream visualization.

In [ ]:
# Remove original HPFa_red objects with high masked-area loss
labels_to_remove = props_hpfa.loc[
    props_hpfa["%_area_cleaned"] >= PCT_AREA_CLEANED_THRESHOLD, "label"
].to_numpy()

hpfa_red_filtered = cell_labels["HPFa_red"].copy()
hpfa_red_filtered[np.isin(hpfa_red_filtered, labels_to_remove)] = 0
props_hpfa_filtered = props_hpfa.loc[
    ~props_hpfa["label"].isin(labels_to_remove)
].copy()

viewer.add_labels(hpfa_red_filtered, name="HPFa_red_filtered")
print(
    f"Removed {len(labels_to_remove)} / {props_hpfa.shape[0]} "
    f"HPFa_red objects (>= {PCT_AREA_CLEANED_THRESHOLD}% area cleaned)"
)

### Visualize extracted features in Napari
Render per-cell volume for A549_green and filtered HPFa_red populations.

Input needed from researcher: verify values and inspect outliers before analysis.

In [ ]:
# Visualize A549_green cell volumes
volume_img = map_df_column_to_labels(
    cell_labels["A549_green"],
    props_a549,
    value_column="A549_green_mean_int",
    colormap="inferno",
    visualize=True,
    layer_name="A549_green_mean_int",
)

In [ ]:
# Visualize filtered HPFa_red cell volumes
volume_img_hpfa = map_df_column_to_labels(
    hpfa_red_filtered,
    props_hpfa_filtered,
    value_column="HPFa_red_mean_int",
    colormap="magma",
    visualize=True,
    layer_name="HPFa_red_mean_int",
)

### Matplotlib visualization

Besides Napari you can use Matplotlib for visualizing your results in a static graph

In [ ]:
plot_prop_to_3d_centroids(
    props_a549,
    value_column="area",
    colormap="inferno",
    nuclei_labels_shape=cell_labels["A549_green"].shape,
    fig_title="A549_green cell volumes",
    ax_labels=None,
    colormap_vmin=None,
    colormap_vmax=None,
    save_fig=False,
    fig_filename=None,
    fig_savepath=None,
    visualize=True
)

In [ ]:
fig = px.histogram(
    props_a549,
    x="area",
    nbins=100,
    title="Histogram of A549_green Cell Area",
    labels={"area": "Cell Area", "count": "Count"},
    opacity=0.7
)
fig.update_traces(marker_color="steelblue", marker_line_color="black", marker_line_width=1)
fig.update_layout(bargap=0.05)
fig.show()